# Portfolio para prop firms (CFD swing) — simulador v3: la escalera desde Rick, sin límite de tiempo

**Qué hace este cuaderno.** Parte del método **exacto** de Rick y cambia **una sola cosa en cada peldaño** hasta llegar al portfolio de la v1, que salió mal. Así se ve qué paso lo estropea.

| Peldaño | Universo | Motor | Tamaño | Costes | Frenos prop |
|---|---|---|---|---|---|
| E1 Rick puro | 5 de Rick (US100, US30, oro, plata, BTC) | Notebook de Rick | 1/N, cada activo entre 0 y 1 | 3,25 pb, sin swap | No |
| E2 + costes CFD | igual | igual | igual | spread + swap | No |
| E3 + frenos prop | igual | igual | igual | spread + swap | Sí |
| E4 + motor corregido | igual | aceleración sin recortar, skew relativo, FDM por correlación | igual | spread + swap | Sí |
| E5 + tamaño con IDM | igual | corregido | IDM, sin tope de 1 | spread + swap | Sí |
| E6 v1 (universo amplio) | 10 CFDs de la v1 | corregido | IDM y pesos por grupo | spread + swap | Sí |
| E7 Rick ×1,5 + frenos | 5 de Rick | Rick | 1/N entre 0 y 1, todo × 1,5 | spread + swap | Sí |
| E8 Rick ×2 + frenos | 5 de Rick | Rick | 1/N entre 0 y 1, todo × 2 | spread + swap | Sí |

**Nuevo en la v3:** las evaluaciones no tienen límite de tiempo (como FTMO). Siguen hasta pasar, romper o acabarse los datos, y solo empiezan las que tienen al menos 2 años de datos por delante. "Sin resolver" pasa a significar "seguía abierta al acabarse los datos".

Todo solo largos (como Rick). Cada peldaño pasa por las evaluaciones FTMO simuladas y por la cuenta fondeada, igual que en la v1.

**Lo único que se fija con datos (y solo con el tramo IS):** en el motor corregido, el FDM entre reglas, el IDM y los scalars del skew relativo.

**Salida:** carpeta nueva en `resultados/` con tablas, gráficas, CSV diario del portfolio y `resumen_prop.txt` (el que me pasas).

> Si `yfinance` no puede descargar, el cuaderno usa datos **sintéticos** y lo avisa con `[DATOS SINTÉTICOS]`: esos números no valen.

## 1 · Configuración

In [ ]:
%matplotlib inline
import os, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- universo: nombre -> (ticker Yahoo, grupo, clase para apalancamiento/costes) ----------------
UNIVERSO = {
    "US500":  ("^GSPC",     "bolsa_usa",  "indice"),
    "US100":  ("^NDX",      "bolsa_usa",  "indice"),
    "GER40":  ("^GDAXI",    "bolsa_intl", "indice"),
    "JP225":  ("^N225",     "bolsa_intl", "indice"),
    "XAUUSD": ("GC=F",      "metales",    "metal"),
    "UKOIL":  ("BZ=F",      "energia",    "energia"),
    "EURUSD": ("EURUSD=X",  "divisas",    "divisa"),
    "USDJPY": ("JPY=X",     "divisas",    "divisa"),
    "AUDUSD": ("AUDUSD=X",  "divisas",    "divisa"),
    "BTCUSD": ("BTC-USD",   "cripto",     "cripto"),
    "US30":   ("^DJI",      "bolsa_usa",  "indice"),     # solo en la cesta de Rick
    "XAGUSD": ("SI=F",      "metales",    "metal"),      # solo en la cesta de Rick
}
CESTA_RICK   = ["US100", "US30", "XAUUSD", "XAGUSD", "BTCUSD"]        # su cesta FTMO: QQQ, Dow, oro, plata, BTC
CESTA_AMPLIA = ["US500", "US100", "GER40", "JP225", "XAUUSD", "UKOIL", "EURUSD", "USDJPY", "AUDUSD", "BTCUSD"]
DATA_START = "2000-01-01"     # calentamiento
EVAL_START = "2018-01-01"     # tu estándar; lo anterior solo calienta el motor (vol de 5 años, EWMAC lento)
IS_FRAC    = 0.70             # IS = primer 70 % de la evaluación (FDM, IDM y skew se fijan aquí)
ANN_DAYS   = 252

# ---------------- reglas de la cuenta (FTMO 2-Step Swing) ----------------
CAPITAL       = 100_000
OBJ_FASE1     = 0.10
OBJ_FASE2     = 0.05
DD_DIARIO     = 0.05          # % del saldo inicial
DD_MAXIMO     = 0.10          # estático
DIAS_MINIMOS  = 4
REF_DIARIA    = "saldo"       # "saldo" (regla FTMO) | "max" (el mayor de saldo y equity)
HORIZONTE     = None          # None = sin límite, como FTMO: la evaluación sigue hasta pasar, romper o acabarse los datos
MIN_FUTURO    = 504           # con HORIZONTE = None, solo se empiezan evaluaciones con al menos 2 años de datos por delante
REPARTO       = 0.80
CUOTA_PCT     = 0.0054        # ~540 EUR en 100k, en % de la cuenta (aprox.)
CADA_COBRO    = 21            # días de bolsa entre cobros en la cuenta fondeada
DIAS_FONDEADA = 252

# ---------------- costes y margen (SUPUESTOS: sustituir por la especificación de MT5) ----------------
SPREAD_BPS = {"indice": 1.5, "metal": 2.0, "energia": 4.0, "divisa": 1.0, "cripto": 8.0}   # por lado
SWAP_LARGO = {"indice": -0.06, "metal": -0.05, "energia": -0.04, "divisa": -0.01, "cripto": -0.15}  # anual
SWAP_CORTO = {"indice": 0.01, "metal": 0.01, "energia": -0.02, "divisa": -0.01, "cripto": -0.15}
APALANC    = {"indice": 15, "metal": 15, "energia": 15, "divisa": 30, "cripto": 1}
MARGEN_MAX = 0.50             # margen usado como mucho el 50 % del equity
TOPE_CRIPTO = 0.25            # nocional máximo de BTC en % del equity
COSTE_RICK_BPS = 3.25         # el coste de su notebook (sin swap)

# ---------------- motor ----------------
FC_TARGET, FC_CAP = 10.0, 20.0
EWMAC_PAIRS = [(8, 32, 5.95), (16, 64, 4.10), (32, 128, 2.79), (64, 256, 1.91)]
PESOS_REGLAS = {"EWMAC": 0.15, "Breakout": 0.15, "Accel": 0.15, "Skew": 0.20}   # motor Rick: tal cual (suman 0,65) x 1,55
FDM_RICK = 1.55
USAR_ATENUACION = True
BUFFER = 0.10                 # inercia: solo se opera si la diferencia supera el 10 % de la posición media
IDM_MAX = 2.5

# ---------------- frenos de riesgo de la cuenta ----------------
FRENO_DD      = True          # τ escalada con la distancia al -10 %
FRENO_DD_MIN  = 0.25          # nunca por debajo del 25 % del riesgo
COLCHON       = True          # mantener al menos 1 % hasta el límite diario si todas las posiciones se mueven 2,5 sigmas en contra a la vez
COLCHON_K     = 2.5
COLCHON_MIN   = 0.01

# ---------------- la escalera ----------------
ESCENARIOS = {
    "E1 Rick puro":         dict(cesta="rick",   motor="rick",     tamano="rick", costes="rick", frenos=False),
    "E2 +costes CFD":       dict(cesta="rick",   motor="rick",     tamano="rick", costes="cfd",  frenos=False),
    "E3 +frenos prop":      dict(cesta="rick",   motor="rick",     tamano="rick", costes="cfd",  frenos=True),
    "E4 +motor corregido":  dict(cesta="rick",   motor="carver",   tamano="rick", costes="cfd",  frenos=True),
    "E5 +tamaño IDM":       dict(cesta="rick",   motor="carver",   tamano="idm",  costes="cfd",  frenos=True),
    "E6 v1 universo amplio":dict(cesta="amplia", motor="carver",   tamano="idm",  costes="cfd",  frenos=True),
    # nuevos en la v3: el tamaño de Rick multiplicado (más rápido), protegido por los frenos
    "E7 Rick x1,5 + frenos": dict(cesta="rick",  motor="rick",     tamano="rick", costes="cfd",  frenos=True, palanca=1.5),
    "E8 Rick x2 + frenos":   dict(cesta="rick",  motor="rick",     tamano="rick", costes="cfd",  frenos=True, palanca=2.0),
}
# con el tamaño de Rick cada activo queda entre 0 y 1, así que hace falta una tau más alta para el mismo riesgo
TAUS_EVAL   = {"rick": [0.20, 0.30, 0.50, 0.75, 1.00], "idm": [0.15, 0.20, 0.25, 0.30, 0.40]}
TAUS_FONDEO = {"rick": [0.10, 0.15, 0.20, 0.30, 0.50], "idm": [0.06, 0.08, 0.10, 0.12, 0.15]}
TAU_REF     = 0.20            # la de su notebook, para el backtest continuo
PASO_INICIOS = 5              # una evaluación simulada que empieza cada 5 días
PASO_INICIOS_FOND = 10        # cuentas fondeadas: cada 10 días (cada una dura un año; más lento)
RIESGO_MAX_FONDEO = 0.05      # al elegir la marcha fondeada: perder la cuenta en 12 meses como mucho un 5 %

RUN_DIR = os.path.join("resultados", f"prop_{dt.datetime.now():%Y%m%d_%H%M%S}")
os.makedirs(RUN_DIR, exist_ok=True)
LOG = []
def log(s=""):
    print(s); LOG.append(str(s))
log(f"Carpeta de esta ejecución: {RUN_DIR}")

## 2 · Datos (OHLC diario)

Todos los instrumentos se alinean en un calendario de lunes a viernes. Un día sin cotización (festivo local) se rellena como día sin movimiento. En BTC, el movimiento del fin de semana cae en el hueco entre el cierre del viernes y la apertura del lunes. Los máximos y mínimos se limpian de impresiones absurdas (más de 5 sigmas fuera de la apertura y el cierre), algo frecuente en las divisas de Yahoo.

In [ ]:
DATOS_REALES = True

def _sintetico(names, n=6900, seed=3):
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    reg = np.zeros(n, int); s = 0
    P = np.array([[0.99, 0.005, 0.005], [0.02, 0.97, 0.01], [0.01, 0.01, 0.98]])
    for t in range(n): s = rng.choice(3, p=P[s]); reg[t] = s
    mkt = np.array([0.0004, -0.0008, 0.0])[reg] + np.array([0.008, 0.02, 0.01])[reg] * rng.standard_normal(n)
    out = {}
    for k, nm in enumerate(names):
        g = UNIVERSO[nm][1]
        beta = {"bolsa_usa": 1.0, "bolsa_intl": 0.8, "metales": -0.1, "energia": 0.4, "divisas": 0.1, "cripto": 0.6}[g]
        iv = {"bolsa_usa": 0.004, "bolsa_intl": 0.007, "metales": 0.009, "energia": 0.018, "divisas": 0.005, "cripto": 0.03}[g]
        trend = np.cumsum(rng.normal(0, 0.00004, n))            # tendencias propias lentas
        r = beta * mkt + trend + iv * rng.standard_normal(n)
        c = 100 * np.exp(np.cumsum(r))
        o = c / np.exp(r * rng.uniform(0.2, 0.6, n))
        rngd = np.abs(r) + iv * np.abs(rng.standard_normal(n))
        h = np.maximum(o, c) * (1 + 0.5 * rngd); l = np.minimum(o, c) * (1 - 0.5 * rngd)
        out[nm] = pd.DataFrame({"Open": o, "High": h, "Low": l, "Close": c}, index=idx)
    return out

def cargar():
    global DATOS_REALES
    data = {}
    try:
        import yfinance as yf
        for nm, (tk, _, _) in UNIVERSO.items():
            df = yf.download(tk, start=DATA_START, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
            df = df[["Open", "High", "Low", "Close"]].dropna()
            df.index = pd.to_datetime(df.index).tz_localize(None)
            if len(df) < 500: raise RuntimeError(f"{tk} vacío")
            data[nm] = df
    except Exception as e:
        DATOS_REALES = False
        log(f"[DATOS SINTÉTICOS] no se pudo descargar ({type(e).__name__}: {e})")
        data = _sintetico(list(UNIVERSO))
    return data

raw = cargar()
cal = pd.bdate_range(max(min(d.index[0] for d in raw.values()), pd.Timestamp(DATA_START)),
                     max(d.index[-1] for d in raw.values()))
O, H, L, Cl = {}, {}, {}, {}
for nm, df in raw.items():
    df = df[df.index.dayofweek < 5].reindex(cal)
    first = df["Close"].first_valid_index()
    c = df["Close"].ffill()
    o = df["Open"].fillna(c); h = df["High"].fillna(c); l = df["Low"].fillna(c)
    o = o.where(o > 0, c); h = h.where(h > 0, c); l = l.where(l > 0, c)
    sd = np.log(c).diff().rolling(60, min_periods=20).std().bfill().fillna(0.01)
    top = np.maximum(o, c) * (1 + 5 * sd); bot = np.minimum(o, c) * (1 - 5 * sd)
    h = np.minimum(np.maximum(h, np.maximum(o, c)), top); l = np.maximum(np.minimum(l, np.minimum(o, c)), bot)
    for X, s in ((O, o), (H, h), (L, l), (Cl, c)):
        X[nm] = s.where(s.index >= first)
O, H, L, Cl = (pd.DataFrame(X) for X in (O, H, L, Cl))
NOMBRES = list(UNIVERSO)
for nm in NOMBRES:
    log(f"{nm:7s} {UNIVERSO[nm][0]:10s} desde {Cl[nm].first_valid_index().date()}")
log(f"Calendario: {cal[0].date()} -> {cal[-1].date()} ({len(cal)} días L-V)")

## 3 · El motor, en dos versiones

- **Rick:** su notebook tal cual: aceleración sobre el EWMAC recortado, skew absoluto con sus scalars, pesos 0,15/0,15/0,15/0,20 × FDM 1,55.
- **Corregido:** aceleración sin recortar, skew relativo (scalars del IS) y FDM por correlación medida en IS.

In [ ]:
idx_eval = Cl.loc[EVAL_START:].index
IS_END = idx_eval[int(len(idx_eval) * IS_FRAC) - 1]
OOS_START = idx_eval[int(len(idx_eval) * IS_FRAC)]
log(f"Evaluación {idx_eval[0].date()} -> {idx_eval[-1].date()} | IS hasta {IS_END.date()} | OOS desde {OOS_START.date()}")

def ewm_std(x, span):
    a = 2.0 / (span + 1.0); m = x.ewm(alpha=a, adjust=False).mean(); m2 = (x * x).ewm(alpha=a, adjust=False).mean()
    return np.sqrt((m2 - m * m).clip(lower=0.0) / ((2.0 - 2.0 * a) / (2.0 - a)))

def vol_stack(close):
    ann = np.sqrt(ANN_DAYS); ret = close.pct_change(fill_method=None).clip(-0.9, 9.0)
    vs_ = ewm_std(ret, 30) * ann; vl_ = vs_.rolling(5 * ANN_DAYS, min_periods=ANN_DAYS).mean()
    vol = pd.Series(np.where(vl_.isna(), vs_, 0.7 * vs_ + 0.3 * vl_), index=close.index)
    return ret, vol, close * vol / ann

def reglas(close, motor):
    ret, vol, sp = vol_stack(close)
    ew = sum(((close.ewm(span=f, adjust=False).mean() - close.ewm(span=s, adjust=False).mean()) / sp * sc).clip(-FC_CAP, FC_CAP)
             for f, s, sc in EWMAC_PAIRS) / 4 * 1.13
    bo = []
    for n, sc, sm in [(40, 0.70, 10), (80, 0.73, 20), (160, 0.74, 40), (320, 0.74, 80)]:
        mn, mx = close.rolling(n).min(), close.rolling(n).max(); rg = mx - mn
        spir = ((close - mn) / rg).where(rg > 0, 0.5)
        bo.append(((spir - 0.5) * 40).ewm(span=sm, adjust=False).mean().mul(sc).clip(-FC_CAP, FC_CAP))
    bo = sum(bo) / 4 * 1.17
    ac = []
    for (f, s, sc), lag, sc2 in zip(EWMAC_PAIRS, [8, 16, 32, 64], [1.87, 1.90, 1.98, 2.05]):
        base = (close.ewm(span=f, adjust=False).mean() - close.ewm(span=s, adjust=False).mean()) / sp * sc
        if motor == "rick": base = base.clip(-FC_CAP, FC_CAP)          # Rick: recorta antes de diferenciar
        ac.append(((base - base.shift(lag)) * sc2).clip(-FC_CAP, FC_CAP))
    ac = sum(ac) / 4 * 1.55
    sk = []
    for w, sm, sc in [(60, 15, 33.3), (120, 30, 37.2), (240, 60, 39.2)]:
        g = ret.rolling(w, min_periods=w // 2).skew()
        if motor == "rick":
            sk.append((-g).ewm(span=sm, adjust=False).mean().mul(sc).clip(-FC_CAP, FC_CAP))   # skew absoluto
        else:
            g = g - g.rolling(5 * ANN_DAYS, min_periods=ANN_DAYS).mean()      # skew relativo
            sk.append((-g).ewm(span=sm, adjust=False).mean())
    out = {"EWMAC": ew, "Breakout": bo, "Accel": ac}
    if motor == "rick": out["Skew"] = (sum(sk) / 3 * 1.18).clip(-FC_CAP, FC_CAP)
    else: out["SkewRaw"] = sk
    return out, ret, vol

def atenuacion(vol):
    p = vol.rolling(1260, min_periods=252).rank(pct=True).ewm(span=10, adjust=False).mean()
    return (1.5 - p).clip(0.5, 1.5).fillna(1.0)

names_r = list(PESOS_REGLAS)
FCM = {}                                   # forecast combinado por motor
for motor in ["rick", "carver"]:
    R, VOL = {}, {}
    for nm in NOMBRES:
        R[nm], _, VOL[nm] = reglas(Cl[nm].dropna(), motor)
    if motor == "carver":
        sk_sc = [10.0 / pd.concat([R[nm]["SkewRaw"][j].loc[:IS_END] for nm in CESTA_AMPLIA]).abs().mean() for j in range(3)]
        for nm in NOMBRES:
            R[nm]["Skew"] = (sum((R[nm]["SkewRaw"][j] * sk_sc[j]).clip(-FC_CAP, FC_CAP) for j in range(3)) / 3 * 1.18).clip(-FC_CAP, FC_CAP)
        wr = np.array([PESOS_REGLAS[k] for k in names_r]); wr = wr / wr.sum()
        pool = pd.concat([pd.concat([R[nm][k] for k in names_r], axis=1, keys=names_r).loc[:IS_END] for nm in CESTA_AMPLIA]).dropna()
        fdm = float(min(2.5, 1 / np.sqrt(wr @ pool.corr().values @ wr)))
    else:
        wr = np.array([PESOS_REGLAS[k] for k in names_r]); fdm = FDM_RICK
    log(f"Motor {motor:6s}: suma de pesos {wr.sum():.2f} x FDM {fdm:.2f} = {wr.sum()*fdm:.2f}")
    FC = {}
    for nm in NOMBRES:
        comb = (pd.concat([R[nm][k] for k in names_r], axis=1).mul(wr, axis=1).sum(axis=1, min_count=4) * fdm).clip(-FC_CAP, FC_CAP)
        if USAR_ATENUACION: comb = (comb * atenuacion(VOL[nm])).clip(-FC_CAP, FC_CAP)
        FC[nm] = comb.reindex(cal)
    FCM[motor] = pd.DataFrame(FC)
VOLd = pd.DataFrame({nm: VOL[nm].reindex(cal) for nm in NOMBRES})

log("\n=== Calibración del forecast combinado (evaluación): motor Rick | motor corregido ===")
for nm in NOMBRES:
    fr_, fc_ = FCM["rick"][nm].loc[EVAL_START:].dropna(), FCM["carver"][nm].loc[EVAL_START:].dropna()
    log(f"{nm:7s} mean|f| {fr_.abs().mean():5.1f} | {fc_.abs().mean():5.1f}   % días largo {(fr_ > 0).mean()*100:4.0f}% | {(fc_ > 0).mean()*100:4.0f}%   vol {VOLd[nm].loc[EVAL_START:].mean()*100:5.1f}%")

## 4 · Del forecast a la posición: las dos formas de dimensionar

Las posiciones se expresan como **fracción del equity** (nocional / equity).

- **Tamaño Rick:** cada activo recibe 1/N del capital y dentro de su parte pesa `clip(forecast/10 × τ/σ, 0, 1)`. Nunca hay más del 100 % invertido en total: no hay apalancamiento.
- **Tamaño IDM (v1):** `forecast/10 × τ × IDM × peso / σ`, sin tope por activo. Puede pasar del 100 % (en la v1, de 1,6 a 2,4 veces el equity).

In [ ]:
GRUPO = {nm: UNIVERSO[nm][1] for nm in NOMBRES}; CLASE = {nm: UNIVERSO[nm][2] for nm in NOMBRES}
EQ_IDX = [nm for nm in NOMBRES if CLASE[nm] == "indice"]

def pesos_instrumento(cesta, por_grupo, FC):
    activo = FC.notna() & VOLd.notna()
    for nm in NOMBRES:
        if nm not in cesta: activo[nm] = False
    W = pd.DataFrame(0.0, index=cal, columns=NOMBRES)
    if not por_grupo:                                                   # 1/N de los activos con datos
        n_act = activo.sum(axis=1).replace(0, np.nan)
        return activo.astype(float).div(n_act, axis=0).fillna(0.0)
    grupos = sorted(set(GRUPO[nm] for nm in cesta))
    n_grupos_act = sum(activo[[nm for nm in NOMBRES if GRUPO[nm] == g]].any(axis=1).astype(int) for g in grupos)
    for g in grupos:
        miembros = [nm for nm in NOMBRES if GRUPO[nm] == g]
        n_act = activo[miembros].sum(axis=1)
        for nm in miembros:
            W[nm] = np.where(activo[nm], 1.0 / n_grupos_act.replace(0, np.nan) / n_act.replace(0, np.nan), 0.0)
    return W.fillna(0.0)

RET_d = pd.DataFrame({nm: Cl[nm].pct_change(fill_method=None) for nm in NOMBRES}).fillna(0.0)

def idm_de(f, W, cesta):
    # retornos de cada subsistema (tau 20 %), solo IS
    sub = ((f.shift(1) / FC_TARGET * (0.20 / VOLd.shift(1))) * RET_d)[cesta]
    Hm = sub.loc[EVAL_START:IS_END].corr().fillna(0.0).to_numpy(copy=True); np.fill_diagonal(Hm, 1.0)
    w = W[cesta].loc[EVAL_START:IS_END].mean().values; w = w / w.sum()
    return float(min(IDM_MAX, 1 / np.sqrt(w @ Hm @ w)))

def preparar(esc):
    """Devuelve una función tau -> array T x K de fracciones (con tau ya aplicada) y una descripción."""
    cesta = CESTA_RICK if esc["cesta"] == "rick" else CESTA_AMPLIA
    f = FCM[esc["motor"]].clip(lower=0)                                   # solo largos
    if esc["tamano"] == "rick":
        W = pesos_instrumento(cesta, False, f)
        k = esc.get("palanca", 1.0)
        def frac(tau):
            return ((f / FC_TARGET) * (tau / VOLd)).clip(0, 1).mul(W).fillna(0.0).values * k
        return frac, f"1/N, cada activo entre 0 y 1, x{k:g}"
    W = pesos_instrumento(cesta, esc["cesta"] == "amplia", f)
    idm = idm_de(f, W, cesta)
    base = ((f / FC_TARGET) * idm * W / VOLd).fillna(0.0).values
    return (lambda tau: base * tau), f"IDM {idm:.2f}"

def aplicar_topes(n):
    n = n.copy()
    btc = [nm for nm in NOMBRES if CLASE[nm] == "cripto"]
    n[btc] = n[btc].clip(-TOPE_CRIPTO, TOPE_CRIPTO)
    lev = np.array([APALANC[CLASE[nm]] for nm in NOMBRES])
    margen = (n.abs() / lev).sum(axis=1)
    escala = (MARGEN_MAX / margen).clip(upper=1.0).fillna(1.0)
    return n.mul(escala, axis=0), (escala < 1).mean()

FRAC = {}
for nombre, esc in ESCENARIOS.items():
    FRAC[nombre], desc = preparar(esc)
    log(f"{nombre:22s} cesta {esc['cesta']:6s} motor {esc['motor']:6s} tamaño {desc}")

## 5 · Backtest continuo de cada peldaño (sin reglas de prop)

Con τ = 20 % (la de su notebook), como si fuera una cuenta propia, más la referencia de comprar y mantener la cesta de Rick a partes iguales. Lo que decide la prop es la columna de **peores días**. Ejecución en la apertura siguiente a la señal.

In [ ]:
DIAS_CAL = pd.Series(cal).diff().dt.days.fillna(1).values
def poner_escenario(esc):
    """Ajusta costes y frenos globales según el peldaño."""
    global SPR, SWL, SWC, FRENO_DD, COLCHON
    if esc["costes"] == "rick":
        SPR = np.full(len(NOMBRES), COSTE_RICK_BPS / 1e4); SWL = np.zeros(len(NOMBRES)); SWC = np.zeros(len(NOMBRES))
    else:
        SPR = np.array([SPREAD_BPS[CLASE[nm]] for nm in NOMBRES]) / 1e4
        SWL = np.array([SWAP_LARGO[CLASE[nm]] for nm in NOMBRES]); SWC = np.array([SWAP_CORTO[CLASE[nm]] for nm in NOMBRES])
    FRENO_DD = COLCHON = esc["frenos"]

def backtest_continuo(N, buffer=BUFFER):
    """N: fracciones decididas al cierre de t; se ejecutan en la apertura de t+1. Inercia en fracción."""
    o = O.values; c = Cl.values; T, K = N.shape
    avgpos = np.abs(N).copy()
    avgpos = pd.DataFrame(avgpos).rolling(250, min_periods=20).mean().bfill().values   # posición media aprox.
    held = np.zeros(K); r = np.zeros(T); gross = np.zeros(T)
    for t in range(1, T):
        cp, op, cc = c[t - 1], o[t], c[t]
        ok = ~np.isnan(cp) & ~np.isnan(op) & ~np.isnan(cc)
        gap = np.where(ok, held * (op / cp - 1), 0.0)
        tgt = N[t - 1]
        mover = np.abs(tgt - held) > buffer * avgpos[t - 1]
        new = np.where(mover & ok, tgt, held)
        cost = np.sum(np.abs(new - held) * SPR)
        intr = np.where(ok, new * (cc / op - 1), 0.0)
        swap = np.sum(np.where(new > 0, np.abs(new) * SWL, np.abs(new) * SWC)) * DIAS_CAL[t] / 365.0
        r[t] = gap.sum() + intr.sum() - cost + swap
        held = np.where(ok, new * (cc / np.where(ok, op, 1)) / (1 + r[t]), new)   # la fracción deriva con el precio
        gross[t] = np.abs(new).sum()
    return pd.Series(r, index=cal), pd.Series(gross, index=cal)

def aplicar_topes_np(n):
    n = n.copy()
    n[:, ES_CRIPTO_] = np.clip(n[:, ES_CRIPTO_], -TOPE_CRIPTO, TOPE_CRIPTO)
    marg = (np.abs(n) / LEV_).sum(axis=1)
    esc = np.minimum(1.0, MARGEN_MAX / np.where(marg > 0, marg, np.inf))
    return n * esc[:, None]
LEV_ = np.array([APALANC[CLASE[nm]] for nm in NOMBRES]); ES_CRIPTO_ = np.array([CLASE[nm] == "cripto" for nm in NOMBRES])

def metricas(r):
    r = r.dropna(); eq = (1 + r).cumprod(); yrs = len(r) / ANN_DAYS; sd = r.std()
    return {"Sharpe": r.mean() / sd * np.sqrt(ANN_DAYS) if sd else np.nan, "CAGR": eq.iloc[-1] ** (1 / yrs) - 1,
            "MaxDD": (eq / eq.cummax() - 1).min(), "Vol": sd * np.sqrt(ANN_DAYS), "PeorDia": r.min(),
            "Dias<-2%": (r < -0.02).mean() * 100, "Dias<-3%": (r < -0.03).mean() * 100}

PER = {"Total": (idx_eval[0], idx_eval[-1]), "IS": (idx_eval[0], IS_END), "OOS": (OOS_START, idx_eval[-1])}
BT = {}
rows = {}
# referencia: comprar y mantener la cesta de Rick a partes iguales (sin costes)
bh = RET_d[CESTA_RICK].loc[EVAL_START:].mean(axis=1)
for p, (a, b) in PER.items(): rows[f"B&H cesta Rick {p}"] = metricas(bh.loc[a:b])
for nombre, esc in ESCENARIOS.items():
    poner_escenario(esc)
    r, gross = backtest_continuo(aplicar_topes_np(FRAC[nombre](TAU_REF)))
    BT[nombre] = (r, gross)
    for p, (a, b) in PER.items():
        m_ = metricas(r.loc[a:b]); m_["Expo"] = gross.loc[a:b].mean(); rows[f"{nombre} {p}"] = m_
tab = pd.DataFrame(rows).T
show = tab.copy()
for col in ["CAGR", "MaxDD", "Vol", "PeorDia"]: show[col] = (show[col] * 100).map(lambda v: f"{v:6.1f}%")
for col in ["Sharpe", "Dias<-2%", "Dias<-3%", "Expo"]: show[col] = show[col].map(lambda v: f"{v:5.2f}")
log(f"\n=== Backtest continuo, tau = {TAU_REF:.0%} (Expo = exposición bruta media, veces el equity) ===")
log(show.to_string())

fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
eqb = (1 + bh).cumprod(); ax[0].plot(eqb.index, eqb, color="gray", ls="--", lw=0.9, label="B&H cesta Rick")
for nombre in ESCENARIOS:
    r = BT[nombre][0].loc[EVAL_START:]; eq = (1 + r).cumprod()
    ax[0].plot(eq.index, eq, lw=1.0, label=nombre); ax[1].plot(eq.index, eq / eq.cummax() - 1, lw=0.8, label=nombre)
ax[0].axvline(OOS_START, color="tab:red", ls=":"); ax[0].set_yscale("log"); ax[0].legend(fontsize=8); ax[0].set_title(f"La escalera desde Rick, tau {TAU_REF:.0%}")
ax[1].set_ylabel("drawdown"); plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "backtest_continuo.png"), dpi=110); plt.show()

## 6 · El simulador de la cuenta

Cada simulación es una cuenta de 100.000 que empieza un día concreto de la historia. Lleva **unidades, precio medio, saldo realizado y flotante** de cada instrumento, porque el límite diario de FTMO se mide desde el saldo de medianoche. Cada día:

1. Al cierre de ayer: posición objetivo = fracción del motor × τ × frenos (drawdown y colchón diario) × equity.
2. Apertura de hoy: el hueco de la noche con la posición vieja; se opera hacia el objetivo (con inercia), pagando spread y realizando beneficio o pérdida de lo que se reduce.
3. Peor caso intradía: todos los instrumentos en su mínimo (si estás largo) o máximo (si estás corto) a la vez.
4. Cierre: equity, swap y comprobación de límites.

Fase 1 (+10 %) → fase 2 (+5 %, cuenta nueva) → cuenta fondeada. Una fase que no se resuelve en `HORIZONTE` días cuenta como "sin resolver".

In [ ]:
o_np, h_np, l_np, c_np = O.values, H.values, L.values, Cl.values
vol_np = VOLd.values
LEV = np.array([APALANC[CLASE[nm]] for nm in NOMBRES])
ES_CRIPTO = np.array([CLASE[nm] == "cripto" for nm in NOMBRES])

def simular(frac, i0, tau, objetivo, dias_max, cobros=False):
    """frac: array T x K con la tau ya aplicada (sin topes); tau solo se usa para estimar el peor día del colchón."""
    K = CAPITAL; E = B = float(K); u = np.zeros(frac.shape[1]); avg = np.zeros_like(u)
    dias_trading = 0; cobrado = 0.0; ultimo_cobro = 0
    avgpos_frac = np.abs(frac[max(0, i0 - 250):i0 + 1]).mean(axis=0) + 1e-12
    for d in range(1, dias_max + 1):
        t = i0 + d
        if t >= len(cal): return "fin_datos", d, cobrado
        cp, op, hh, ll, cc = c_np[t - 1], o_np[t], h_np[t], l_np[t], c_np[t]
        ok = np.isfinite(cp) & np.isfinite(op) & np.isfinite(cc)
        cpz, opz, ccz = np.where(ok, cp, 0), np.where(ok, op, 0), np.where(ok, cc, 0)
        # --- decisión al cierre de ayer ---
        escala = 1.0
        if FRENO_DD:
            escala = min(1.0, max(FRENO_DD_MIN, (E - (1 - DD_MAXIMO) * K) / (DD_MAXIMO * K)))
        n = frac[t - 1] * escala
        n = np.where(ES_CRIPTO, np.clip(n, -TOPE_CRIPTO, TOPE_CRIPTO), n)
        marg = np.sum(np.abs(n) / LEV)
        if marg > MARGEN_MAX: n = n * MARGEN_MAX / marg
        if COLCHON:
            ref = B if REF_DIARIA == "saldo" else max(B, E)
            hueco = E - (ref - DD_DIARIO * K)                           # distancia al límite diario de mañana
            # peor día: COLCHON_K sigmas de cada posición, todas en contra a la vez (correlación 1)
            peor = COLCHON_K * np.nansum(np.abs(n) * vol_np[t - 1]) / 16 * E
            if peor > 0 and hueco - COLCHON_MIN * K < peor:
                n = n * max(0.0, (hueco - COLCHON_MIN * K) / peor)
        tgt_u = np.where(ok, n * E / np.where(ok, cp, 1), u)
        buf_u = np.where(ok, BUFFER * avgpos_frac * E / np.where(ok, cp, 1), 0)
        new_u = np.where(np.abs(tgt_u - u) > buf_u, tgt_u, u)
        ref_dia = B if REF_DIARIA == "saldo" else max(B, E)                # saldo de medianoche
        # --- apertura: hueco nocturno y operación ---
        gap = np.sum(u * (opz - cpz))
        du = new_u - u
        coste = np.sum(np.abs(du) * opz * SPR)
        realiz = 0.0
        for k in np.nonzero(du)[0]:
            if u[k] != 0 and np.sign(du[k]) != np.sign(u[k]):          # reduce o gira
                cerr = min(abs(du[k]), abs(u[k])) * np.sign(u[k])
                realiz += cerr * (opz[k] - avg[k])
                if abs(du[k]) > abs(u[k]): avg[k] = opz[k]               # gira: nuevo precio medio
            elif new_u[k] != 0:                                           # aumenta
                avg[k] = (u[k] * avg[k] + du[k] * opz[k]) / new_u[k] if u[k] != 0 else opz[k]
        if np.any(du != 0): dias_trading += 1
        u = new_u
        E_open = E + gap - coste
        # --- peor caso intradía ---
        adverso = np.where(u > 0, np.where(ok, ll, 0), np.where(ok, hh, 0))
        E_low = E_open + np.sum(u * (adverso - opz))
        # --- cierre ---
        swap = np.sum(np.where(u > 0, np.abs(u * ccz) * SWL, np.abs(u * ccz) * SWC)) * DIAS_CAL[t] / 365.0
        E = E_open + np.sum(u * (ccz - opz)) + swap
        B = B + realiz - coste + swap
        if E_low < ref_dia - DD_DIARIO * K: return "rotura_diaria", d, cobrado
        if E_low < (1 - DD_MAXIMO) * K:    return "rotura_maxima", d, cobrado
        if not cobros and E >= K * (1 + objetivo) and dias_trading >= DIAS_MINIMOS:
            return "pasa", d, cobrado
        if cobros and d - ultimo_cobro >= CADA_COBRO and E > K:
            pago = E - K; cobrado += REPARTO * pago / K
            esc = K / E; u = u * esc; avg = np.where(u != 0, ccz, 0.0)      # se realiza todo y se reabre al precio actual
            E = B = float(K); ultimo_cobro = d
    return ("sin_resolver" if not cobros else "sobrevive"), dias_max, cobrado

def evaluacion(frac, i0, tau):
    h1 = HORIZONTE or (len(cal) - i0 - 1)
    r1, d1, _ = simular(frac, i0, tau, OBJ_FASE1, h1)
    if r1 != "pasa": return r1, d1, 1
    h2 = HORIZONTE or (len(cal) - (i0 + d1) - 1)
    r2, d2, _ = simular(frac, i0 + d1, tau, OBJ_FASE2, h2)
    return r2, d1 + d2, 2

t_eval0 = cal.get_loc(idx_eval[0]); t_is = cal.get_loc(IS_END); T = len(cal)
INICIOS_EVAL = [i for i in range(t_eval0, T - (2 * HORIZONTE if HORIZONTE else MIN_FUTURO), PASO_INICIOS)]
INICIOS_FOND = [i for i in range(t_eval0, T - DIAS_FONDEADA, PASO_INICIOS_FOND)]
log(f"\nEvaluaciones simuladas por combinación: {len(INICIOS_EVAL)} | cuentas fondeadas: {len(INICIOS_FOND)}")

## 7 · Marcha 1: evaluación (fases 1 + 2)

Para cada peldaño y cada τ: % que pasa las dos fases, % que rompe el límite diario o el máximo, % sin resolver y días medianos hasta pasar. Separado por evaluaciones que **empiezan** en IS y en OOS.

In [ ]:
t0 = time.time()
res_eval = []
for m, esc in ESCENARIOS.items():
    poner_escenario(esc)
    for tau in TAUS_EVAL[esc["tamano"]]:
        fr = FRAC[m](tau)
        for i0 in INICIOS_EVAL:
            res, dias, fase = evaluacion(fr, i0, tau)
            res_eval.append({"modo": m, "tau": tau, "inicio": cal[i0], "tramo": "IS" if i0 <= t_is else "OOS",
                             "resultado": res, "dias": dias, "fase_fin": fase})
RE = pd.DataFrame(res_eval)
log(f"(simulación de evaluaciones: {time.time()-t0:.0f} s)")

def resumen_eval(df):
    g = df.groupby(["modo", "tau"])
    out = pd.DataFrame({
        "Pasa %": g.apply(lambda x: (x.resultado == "pasa").mean() * 100),
        "Rot.diaria %": g.apply(lambda x: (x.resultado == "rotura_diaria").mean() * 100),
        "Rot.máx %": g.apply(lambda x: (x.resultado == "rotura_maxima").mean() * 100),
        "Sin resolver %": g.apply(lambda x: (x.resultado == "sin_resolver").mean() * 100),
        "Días mediana (pasa)": g.apply(lambda x: x.loc[x.resultado == "pasa", "dias"].median()),
        "Días p75 (pasa)": g.apply(lambda x: x.loc[x.resultado == "pasa", "dias"].quantile(0.75)),
        "Falla en fase 2 %": g.apply(lambda x: ((x.resultado != "pasa") & (x.fase_fin == 2)).mean() * 100),
    })
    return out.round(1)

for tramo in ["IS", "OOS"]:
    log(f"\n=== Evaluación FTMO 2-Step — inicios en {tramo} ===")
    log(resumen_eval(RE[RE.tramo == tramo]).to_string())
TOT_EVAL = resumen_eval(RE)
log("\n=== Evaluación — todos los inicios ===")
log(TOT_EVAL.to_string())
TOT_EVAL.to_csv(os.path.join(RUN_DIR, "evaluacion_resumen.csv"))

fig, ax = plt.subplots(figsize=(10, 4.8))
for m in ESCENARIOS:
    s = TOT_EVAL.loc[m, "Pasa %"]; ax.plot(range(len(s)), s.values, marker="o", label=m)
ax.axhline(33.3, color="gray", ls="--", lw=0.8); ax.text(0, 34, "sin ventaja (azar) ≈ 33 %", color="gray", fontsize=8)
ax.set_xticks(range(5)); ax.set_xticklabels(["τ1", "τ2", "τ3", "τ4", "τ5"])
ax.set_xlabel("tau de evaluación, de menor a mayor (ver tabla)"); ax.set_ylabel("% que pasa las dos fases"); ax.set_title("Probabilidad de pasar por peldaño")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "prob_pasar.png"), dpi=110); plt.show()

## 8 · Marcha 2: cuenta fondeada

Cada cuenta empieza un día de la historia con τ baja y dura `DIAS_FONDEADA` días. Cada `CADA_COBRO` días, si hay beneficio, se cobra el 80 % y la cuenta vuelve a 100.000. Se mide la probabilidad de perderla y lo cobrado (en % de la cuenta).

In [ ]:
t0 = time.time()
res_f = []
for m, esc in ESCENARIOS.items():
    poner_escenario(esc)
    for tau in TAUS_FONDEO[esc["tamano"]]:
        fr = FRAC[m](tau)
        for i0 in INICIOS_FOND:
            res, dias, cobrado = simular(fr, i0, tau, 0.0, DIAS_FONDEADA, cobros=True)
            res_f.append({"modo": m, "tau": tau, "tramo": "IS" if i0 <= t_is else "OOS",
                          "resultado": res, "dias": dias, "cobrado": cobrado})
RF = pd.DataFrame(res_f)
log(f"(simulación de cuentas fondeadas: {time.time()-t0:.0f} s)")

def resumen_fond(df):
    g = df.groupby(["modo", "tau"])
    return pd.DataFrame({
        "Pierde la cuenta %": g.apply(lambda x: (x.resultado != "sobrevive").mean() * 100),
        "Cobrado medio % cuenta": g.apply(lambda x: x.cobrado.mean() * 100),
        "Cobrado mediana %": g.apply(lambda x: x.cobrado.median() * 100),
        "Cobrado si sobrevive %": g.apply(lambda x: x.loc[x.resultado == "sobrevive", "cobrado"].mean() * 100),
    }).round(2)

for tramo in ["IS", "OOS"]:
    log(f"\n=== Cuenta fondeada (12 meses) — inicios en {tramo} ===")
    log(resumen_fond(RF[RF.tramo == tramo]).to_string())
TOT_F = resumen_fond(RF)
log("\n=== Cuenta fondeada — todos los inicios ===")
log(TOT_F.to_string())
TOT_F.to_csv(os.path.join(RUN_DIR, "fondeada_resumen.csv"))

## 9 · Las dos marchas juntas: valor esperado

Valor esperado de comprar una evaluación, en % del tamaño de la cuenta, para cada combinación (τ evaluación, τ fondeada) dentro de cada modo:

**VE = P(pasar) × (cobrado medio en 12 meses + cuota devuelta) − cuota**

Es una aproximación: no cuenta el tiempo hasta pasar ni el plan de escalado.

In [ ]:
ve_rows = []
for m, esc in ESCENARIOS.items():
    for te in TAUS_EVAL[esc["tamano"]]:
        p = TOT_EVAL.loc[(m, te), "Pasa %"] / 100
        for tf in TAUS_FONDEO[esc["tamano"]]:
            cobr = TOT_F.loc[(m, tf), "Cobrado medio % cuenta"] / 100
            pierde = TOT_F.loc[(m, tf), "Pierde la cuenta %"] / 100
            ve = p * (cobr + CUOTA_PCT) - CUOTA_PCT
            ve_rows.append({"modo": m, "tau_eval": te, "tau_fondeo": tf, "P_pasa %": p * 100,
                            "Pierde fondeada %": pierde * 100, "Cobrado 12m %": cobr * 100, "VE % cuenta": ve * 100})
VE = pd.DataFrame(ve_rows)
# el VE a 12 meses no castiga perder la cuenta (se pierden cobros futuros): por eso se limita ese riesgo
validas = VE[VE["Pierde fondeada %"] <= RIESGO_MAX_FONDEO * 100]
mejor = validas.sort_values("VE % cuenta", ascending=False).groupby("modo").head(1)
log(f"\n=== Mejor par de marchas por modo (VE, perder la fondeada <= {RIESGO_MAX_FONDEO:.0%}) ===")
log(mejor.round(2).to_string(index=False))
log("\n=== Sin ese límite (solo para comparar) ===")
log(VE.sort_values("VE % cuenta", ascending=False).groupby("modo").head(1).round(2).to_string(index=False))
log("\n(VE en euros para una cuenta de 100k = VE % × 1.000)")

# resumen de la escalera
esc_rows = {}
for m in ESCENARIOS:
    t = tab.loc[f"{m} Total"]; best = TOT_EVAL.loc[m]["Pasa %"]
    esc_rows[m] = {"Sharpe": t.Sharpe, "CAGR %": t.CAGR * 100, "MaxDD %": t.MaxDD * 100, "Peor día %": t.PeorDia * 100,
                   "Días<-3% (%)": t["Dias<-3%"], "Expo": t.Expo, "Mejor pasa %": best.max(), "con tau": best.idxmax(),
                   "Pasa % OOS (esa tau)": resumen_eval(RE[(RE.tramo == "OOS") & (RE.modo == m)]).loc[(m, best.idxmax()), "Pasa %"]}
log("\n=== RESUMEN DE LA ESCALERA (backtest con tau 20 % + mejor evaluación) ===")
log(pd.DataFrame(esc_rows).T.round(2).to_string())
VE.round(3).to_csv(os.path.join(RUN_DIR, "valor_esperado.csv"), index=False)

## 10 · Exportar

In [ ]:
for m in ESCENARIOS:
    r, gross = BT[m]
    pd.DataFrame({"date": cal.strftime("%Y-%m-%d"), "strategy_return": r.values, "gross_exposure": gross.values}) \
      .loc[cal >= pd.Timestamp(EVAL_START)].to_csv(os.path.join(RUN_DIR, f"portfolio_{m.split()[0]}_tau{int(TAU_REF*100)}_daily.csv"), index=False)
RE.to_csv(os.path.join(RUN_DIR, "evaluaciones_detalle.csv"), index=False)
ctx = f"""CONTEXTO — portfolio prop firm, simulador v2 (escalera desde Rick)
1. Calendario: L-V, decisión al cierre, ejecución en la apertura siguiente. Universo: {', '.join(NOMBRES)}.
2. Periodos: evaluación {idx_eval[0].date()} -> {idx_eval[-1].date()}; IS hasta {IS_END.date()}; OOS desde {OOS_START.date()}.
3. Costes: spread {SPREAD_BPS} pb por lado; swaps largo {SWAP_LARGO}, corto {SWAP_CORTO} (supuestos).
4. Reglas: FTMO 2-Step Swing, objetivos {OBJ_FASE1:.0%}/{OBJ_FASE2:.0%}, diario {DD_DIARIO:.0%} desde {REF_DIARIA}, máximo {DD_MAXIMO:.0%} estático, horizonte {HORIZONTE or 'sin límite'} por fase.
Escenarios: {list(ESCENARIOS)}. Inicio cada {PASO_INICIOS} días. Datos reales: {DATOS_REALES}.
"""
log("\n" + ctx)
if not DATOS_REALES: log("AVISO: DATOS SINTÉTICOS. Los números no valen; ejecuta con yfinance.")
open(os.path.join(RUN_DIR, "contexto_prop.txt"), "w", encoding="utf-8").write(ctx)
open(os.path.join(RUN_DIR, "resumen_prop.txt"), "w", encoding="utf-8").write("\n".join(LOG))
print("Guardado en", RUN_DIR, os.listdir(RUN_DIR))